In [2]:
import cv2
import os
import time
from datetime import datetime
from ultralytics import YOLO


In [ ]:
# CẤU HÌNH HỆ THỐNG 
VIOLATION_THRESHOLD = 10    # 10 giây để ghi tội
SAVE_INTERVAL = 7200       
CLEANUP_INTERVAL = 1800    
save_path = "hospital_violations"
os.makedirs(save_path, exist_ok=True)

model = YOLO('best.pt')
tracker_history = {} # Lưu: {id: {'start_violation': ts, 'last_saved': ts, 'last_seen': ts}}

cap = cv2.VideoCapture(0) 

last_cleanup_time = time.time()

In [ ]:
while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    current_ts = time.time()
    
    # TẮT LOG HỆ THỐNG: verbose=False để không hiện "Object Detection..." nặng máy
    results = model.track(frame, persist=True, conf=0.4, tracker="bytetrack.yaml", verbose=False)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        clss = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)

        for box, cls, conf, obj_id in zip(boxes, clss, confs, ids):
            label_name = model.names[cls]
            
            
            print(f"ID: {obj_id: <3} | Nhãn: {label_name: <20} | Độ tin cậy: {conf:.2%}")

            # Khởi tạo hoặc cập nhật thời gian nhìn thấy
            if obj_id not in tracker_history:
                tracker_history[obj_id] = {'start_violation': current_ts, 'last_saved': 0, 'last_seen': current_ts}
            else:
                tracker_history[obj_id]['last_seen'] = current_ts

            # LOGIC GHI TỘI (Class 1: no_mask, Class 2: incorrect)
            if cls in [1, 2]:
                violation_duration = current_ts - tracker_history[obj_id]['start_violation']
                
                # Kiểm tra mốc 10 giây
                if violation_duration >= VIOLATION_THRESHOLD:
                    time_since_last_save = current_ts - tracker_history[obj_id]['last_saved']
                    
                    if time_since_last_save >= SAVE_INTERVAL:
                        x1, y1, x2, y2 = map(int, box)
                        # Crop vùng mặt vi phạm
                        face_crop = frame[max(0,y1):min(frame.shape[0],y2), max(0,x1):min(frame.shape[1],x2)]
                        
                        if face_crop.size > 0:
                            timestamp_str = datetime.now().strftime("%H%M%S")
                            filename = f"ID{obj_id}_{label_name}_{timestamp_str}.jpg"
                            full_path = os.path.join(save_path, filename)
                            
                            # GHI FILE
                            cv2.imwrite(full_path, face_crop)
                            tracker_history[obj_id]['last_saved'] = current_ts
                            print(f"ĐÃ GHI TỘI ID {obj_id} (Vi phạm >10s): {filename}")
            else:
                # Nếu đeo khẩu trang đúng, reset mốc thời gian bắt đầu vi phạm
                tracker_history[obj_id]['start_violation'] = current_ts

    
    if current_ts - last_cleanup_time > 300:
        ids_to_delete = [i for i, d in tracker_history.items() if current_ts - d['last_seen'] > CLEANUP_INTERVAL]
        for i in ids_to_delete: del tracker_history[i]
        last_cleanup_time = current_ts

   
    cv2.imshow("Hospital Monitor", results[0].plot())
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

ID: 1   | Nhãn: without_mask         | Độ tin cậy: 66.64%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 70.92%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 65.03%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 71.80%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 71.80%
ID: 2   | Nhãn: without_mask         | Độ tin cậy: 64.93%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 78.03%
ID: 2   | Nhãn: without_mask         | Độ tin cậy: 64.59%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 78.03%
ID: 2   | Nhãn: without_mask         | Độ tin cậy: 64.59%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 84.57%
ID: 2   | Nhãn: without_mask         | Độ tin cậy: 81.38%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 81.79%
ID: 2   | Nhãn: without_mask         | Độ tin cậy: 64.25%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 81.79%
ID: 2   | Nhãn: without_mask         | Độ tin cậy: 64.25%
ID: 1   | Nhãn: without_mask         | Độ tin cậy: 77.74%
ID: 2   | Nhãn